In [1]:
!git clone https://github.com/WeilunWang/semantic-diffusion-model
%cd semantic-diffusion-model
!pip install -q flask flask-ngrok pyngrok numpy Pillow torch torchvision

fatal: destination path 'semantic-diffusion-model' already exists and is not an empty directory.
/content/semantic-diffusion-model


check if still exists
https://drive.google.com/file/d/1O8Avsvfc8rP9LIt5tkJxowMTpi1nYiik/view?usp=drive_link


In [2]:
!pip install -q gdown

import gdown
import os

os.makedirs("./net_sdm_ade20k/sdm_ade20k", exist_ok=True)

gdown.download(
    id="1O8Avsvfc8rP9LIt5tkJxowMTpi1nYiik",
    output = "./net_sdm_ade20k/sdm_ade20k/ema_0.9999_best.pt",
    quiet=False
)
!pip install -q flask flask-ngrok pyngrok numpy Pillow  gdown

!pip install torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121


Downloading...
From (original): https://drive.google.com/uc?id=1O8Avsvfc8rP9LIt5tkJxowMTpi1nYiik
From (redirected): https://drive.google.com/uc?id=1O8Avsvfc8rP9LIt5tkJxowMTpi1nYiik&confirm=t&uuid=80fec0b6-5d5c-4d74-8dd4-abb9415134a3
To: /content/semantic-diffusion-model/net_sdm_ade20k/sdm_ade20k/ema_0.9999_best.pt
100%|██████████| 2.45G/2.45G [00:20<00:00, 122MB/s]


Looking in indexes: https://download.pytorch.org/whl/cu121


In [5]:

import zipfile
import sys
import os


import torch

import base64
import io
import shutil

file_path = "./net_sdm_ade20k/sdm_ade20k/ema_0.9999_best.pt"
extract_dir = "./extracted_checkpoint"

import struct


outer_zip = "./net_sdm_ade20k/sdm_ade20k/ema_0.9999_best.pt"
extract_dir = "./extracted_inner"
os.makedirs(extract_dir, exist_ok=True)

print("extracting inner file...")
with zipfile.ZipFile(outer_zip, 'r') as zf:
    zf.extractall(extract_dir)

inner_file = os.path.join(extract_dir, "sdm_ade20k", "ema_0.9999_best.pt")

try:
    print("standard torch.load...")
    checkpoint = torch.load(inner_file, map_location='cpu', weights_only=False)
    print(f"Checkpoint type: {type(checkpoint)}")
    if isinstance(checkpoint, dict):
        print(f"Keys: {list(checkpoint.keys())[:10]}")
    print(f"Model keys: {list(checkpoint.keys())}")
except Exception as e:
    print(f"failed: {e}")
"""
try:
    print("PyTorch Lightning format...")
    checkpoint = torch.load(inner_file, map_location='cpu')
    if 'state_dict' in checkpoint:
        print("Found 'state_dict' key - PyTorch Lightning checkpoint!")
        state_dict = checkpoint['state_dict']
        print(f"State dict keys (first 10): {list(state_dict.keys())[:10]}")
    elif 'model' in checkpoint:
        state_dict = checkpoint['model']
        print(f"Model keys (first 10): {list(state_dict.keys())[:10]}")
except Exception as e:
    print(f"failed: {e}")
"""

if 'checkpoint' in dir():
    final_path = "./net_sdm_ade20k/sdm_ade20k/ema_0.9999_best_WORKING.pt"
    shutil.copy2(inner_file, final_path)
    print(f"Copied working checkpoint to: {final_path}")



extracting inner file...
standard torch.load...
Checkpoint type: <class 'collections.OrderedDict'>
Keys: ['time_embed.0.weight', 'time_embed.0.bias', 'time_embed.2.weight', 'time_embed.2.bias', 'input_blocks.0.0.weight', 'input_blocks.0.0.bias', 'input_blocks.1.0.in_layers.0.weight', 'input_blocks.1.0.in_layers.0.bias', 'input_blocks.1.0.in_layers.2.weight', 'input_blocks.1.0.in_layers.2.bias']
Model keys: ['time_embed.0.weight', 'time_embed.0.bias', 'time_embed.2.weight', 'time_embed.2.bias', 'input_blocks.0.0.weight', 'input_blocks.0.0.bias', 'input_blocks.1.0.in_layers.0.weight', 'input_blocks.1.0.in_layers.0.bias', 'input_blocks.1.0.in_layers.2.weight', 'input_blocks.1.0.in_layers.2.bias', 'input_blocks.1.0.emb_layers.1.weight', 'input_blocks.1.0.emb_layers.1.bias', 'input_blocks.1.0.out_layers.0.weight', 'input_blocks.1.0.out_layers.0.bias', 'input_blocks.1.0.out_layers.3.weight', 'input_blocks.1.0.out_layers.3.bias', 'input_blocks.2.0.in_layers.0.weight', 'input_blocks.2.0.in_lay

In [7]:
!pip install pyngrok -q
import torch

import torch.nn.functional as F
import base64
import io
import numpy as np
import traceback
from PIL import Image
from flask import Flask, request, jsonify
from pyngrok import ngrok
import time
import sys

sys.path.insert(0, '/content/semantic-diffusion-model')

ngrok.set_auth_token("3CgSO6XMliuZ8xbnhIHmsi4QZSQ_3xALJqRx1jruTU4xuV6v2")

ngrok.kill()
time.sleep(1)

from guided_diffusion.script_util import (
    model_and_diffusion_defaults,
    create_model_and_diffusion,
    args_to_dict,
)
from types import SimpleNamespace



args = model_and_diffusion_defaults()
args.update({
    "attention_resolutions": "32,16,8",
    "class_cond": True,
    "diffusion_steps": 1000,
    "image_size": 256,
    "learn_sigma": True,
    "noise_schedule": "linear",
    "num_channels": 256,
    "num_head_channels": 64,
    "num_res_blocks": 2,
    "resblock_updown": True,
    "use_fp16": True,
    "use_scale_shift_norm": True,
    "timestep_respacing": "100",
    "dropout": 0.0,
    "use_checkpoint": True,
    "num_classes": 151,
    "dataset_mode": "ade20k",
    "no_instance": True,
})

args_ns = SimpleNamespace(**args)


model, diffusion = create_model_and_diffusion(
    **args_to_dict(args_ns, model_and_diffusion_defaults().keys())
)

checkpoint_path = "./net_sdm_ade20k/sdm_ade20k/ema_0.9999_best_WORKING.pt"
state_dict = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
model.load_state_dict(state_dict)

model = model.cuda()
model = model.half()
model.eval()


app = Flask(__name__)

def decode_image(b64_str: str) -> np.ndarray:
    data = base64.b64decode(b64_str)
    return np.array(Image.open(io.BytesIO(data)).convert("RGB"))

def encode_image(arr: np.ndarray) -> str:
    img = Image.fromarray(arr.astype(np.uint8))
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()

@app.route("/", methods=["POST"])
def generate():
    try:
        data = request.json
        if not data or "mask" not in data:
            return jsonify({"error": "Missing 'mask' field"}), 400

        mask_b64 = data["mask"]
        mask_bytes = base64.b64decode(mask_b64)
        mask_pil = Image.open(io.BytesIO(mask_bytes)).convert("L")
        mask_np = np.array(mask_pil)

        mask_pil_resized = Image.fromarray(mask_np).resize((256, 256), Image.NEAREST)
        mask_np_resized = np.array(mask_pil_resized)
        num_classes = 151
        mask_np_resized = np.clip(mask_np_resized, 0, num_classes - 1)

        label_indices = torch.from_numpy(mask_np_resized).long()
        label_indices = label_indices.unsqueeze(0)
        label_one_hot = F.one_hot(label_indices, num_classes=num_classes)
        label_one_hot = label_one_hot.permute(0, 3, 1, 2).float()
        label = label_one_hot.cuda().half()

        with torch.no_grad():
            with torch.cuda.amp.autocast(dtype=torch.float16):
                sample = diffusion.p_sample_loop(
                    model,
                    (1, 3, 256, 256),
                    clip_denoised=True,
                    model_kwargs={"y": label},
                    progress=False,
                )

        sample_float = sample.float()
        out = ((sample_float + 1) / 2 * 255).clamp(0, 255)
        out_np = out.squeeze(0).permute(1, 2, 0).cpu().numpy().astype(np.uint8)
        result_b64 = encode_image(out_np)

        print("rquest completed successfully!")
        return jsonify({"image": result_b64})

    except Exception as e:
        print("generate error ук:")
        traceback.print_exc()
        return jsonify({"error": str(e)}), 500

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "healthy", "model_loaded": True})

tunnel = ngrok.connect(5000)
public_url = tunnel.public_url
print(f"PUBLIC URL: {public_url}\n")

app.run(host="0.0.0.0", port=5000)

PUBLIC URL: https://send-krypton-straining.ngrok-free.dev

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
